# Прогнозируем задержки самолетов

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [3]:
RANDOM_STATE = 111
DATASET_PATH = 'https://raw.githubusercontent.com/evgpat/edu_stepik_practical_ml/main/datasets/flight_delays_train.csv'

In [4]:
data = pd.read_csv(DATASET_PATH)

x = data.drop('dep_delayed_15min', axis=1)
y = data['dep_delayed_15min'] == 'Y'

x.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732
1,c-4,c-20,c-3,1548,US,PIT,MCO,834
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423


Создайте список номеров колонок с категориальными признаками для бустингов

## Quiz
Какой длины получился список?

(подсказка: колонка `DepTime` числовая)

In [5]:
# your code here
cat_features = x.select_dtypes(include='object').columns.to_list()
cat_features

['Month', 'DayofMonth', 'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest']

Разобъем данные на обучение и контроль

In [6]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=RANDOM_STATE)

In [7]:
xtrain.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
41207,c-4,c-18,c-1,1457,CO,EWR,TPA,998
28283,c-11,c-1,c-2,1225,UA,DEN,BOS,1754
34619,c-6,c-16,c-5,1650,YV,IAD,CAE,401
8789,c-5,c-18,c-4,923,AA,SLC,DFW,988
38315,c-2,c-14,c-2,1839,AA,STL,SAN,1558


## Модели с параметрами по умолчанию

Обучите CatBoost с гиперпараметрами по умолчанию.

## Quiz
Чему равен ROC-AUC на тестовых данных? Ответ округлите до сотых.

In [8]:
# your code here
cb = CatBoostClassifier(random_seed=RANDOM_STATE, verbose=False)
cb.fit(xtrain, ytrain, cat_features=cat_features)

cb_pred = cb.predict_proba(xtest)[:,1]
roc_auc_score(ytest, cb_pred)

0.7656748885779602

Обучите LightGBM с гиперпараметрами по умолчанию.

## Quiz
Чему равен ROC-AUC на тестовых данных? Ответ округлите до сотых.

In [9]:
for c in x.columns:
    col_type = x[c].dtype
    if col_type == 'object' or col_type.name == 'category':
        xtrain[c] = xtrain[c].astype('category')
        xtest[c] = xtest[c].astype('category')

In [10]:
# your code here
lgbm = LGBMClassifier(random_state=RANDOM_STATE)
lgbm.fit(xtrain, ytrain)

lgbm_pred = lgbm.predict_proba(xtest)[:,1]
roc_auc_score(ytest, lgbm_pred)

[LightGBM] [Info] Number of positive: 14346, number of negative: 60654
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000984 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 75000, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.191280 -> initscore=-1.441714
[LightGBM] [Info] Start training from score -1.441714


0.7341149074685321

## Optuna

Выделим дополнительную валидационную выборку.

In [11]:
xtrain_new, xval, ytrain_new, yval = train_test_split(xtrain, ytrain, test_size=0.25, random_state=RANDOM_STATE)

Создайте функцию objective_lgbm, в которой среди гиперпараметров

* num_leaves = trial.suggest_int("num_leaves", 10, 100)
* n_estimators = trial.suggest_int("n_estimators", 10, 1000)

подберите оптимальные, обучая LGBM на Xtrain_new, ytrain_new и проверяя качество (ROC-AUC) на Xval.

Используйте 30 эпох обучения Optuna.


In [12]:
import optuna

# your code here
def objective_lgbm(trial):
    num_leaves = trial.suggest_int('num_leaves', 10, 100)
    n_estimators = trial.suggest_int('n_estimators', 10, 1000)

    model = LGBMClassifier(num_leaves=num_leaves, n_estimators=n_estimators, random_state=RANDOM_STATE, verbose=-1)
    model.fit(xtrain_new, ytrain_new)
    pred = model.predict_proba(xval)[:,1]

    score = roc_auc_score(yval, pred)

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective_lgbm, n_trials=30)

[I 2026-04-08 19:54:17,733] A new study created in memory with name: no-name-39f40db1-a3fc-4526-956d-03ddaf7b6619
[I 2026-04-08 19:54:19,778] Trial 0 finished with value: 0.7182771043281786 and parameters: {'num_leaves': 35, 'n_estimators': 780}. Best is trial 0 with value: 0.7182771043281786.
[I 2026-04-08 19:54:20,992] Trial 1 finished with value: 0.7182382734884004 and parameters: {'num_leaves': 53, 'n_estimators': 361}. Best is trial 0 with value: 0.7182771043281786.
[I 2026-04-08 19:54:22,486] Trial 2 finished with value: 0.7150830530447179 and parameters: {'num_leaves': 24, 'n_estimators': 863}. Best is trial 0 with value: 0.7182771043281786.
[I 2026-04-08 19:54:24,554] Trial 3 finished with value: 0.7183431989858152 and parameters: {'num_leaves': 91, 'n_estimators': 533}. Best is trial 3 with value: 0.7183431989858152.
[I 2026-04-08 19:54:28,466] Trial 4 finished with value: 0.7186117622103896 and parameters: {'num_leaves': 99, 'n_estimators': 857}. Best is trial 4 with value: 0

In [13]:
study.best_params

{'num_leaves': 66, 'n_estimators': 29}

Обучите модель с найденными гиперпараметрами на Xtrain, ytrain и оцените ROC-AUC на тестовых данных.

In [14]:
# your code here
best_lgbm = LGBMClassifier(**study.best_params, random_state=RANDOM_STATE)
best_lgbm.fit(xtrain, ytrain)

final_preds = best_lgbm.predict_proba(xtest)[:, 1]
final_roc_auc = roc_auc_score(ytest, final_preds)

print(f"Final LightGBM ROC-AUC: {final_roc_auc:.2f}")
print(f"Количество деревьев: {study.best_params['n_estimators']}")

Final LightGBM ROC-AUC: 0.73
Количество деревьев: 29


## Quiz

Чему равно количество деревьев в LGBM после подбора гиперпараметров?

## Работа над улучшением модели

* Попробуйте при помощи Optuna подобрать и другие гиперпарамтеры
* Также подберите гиперпараметры у CatBoost (а не только у LightGBM)

In [15]:
# your code here
def objective_lgbm_pro(trial):
    params = {'num_leaves': trial.suggest_int('num_leaves', 10, 100),
              'n_estimators': trial.suggest_int('n_estimators', 10, 1000),
              'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
              'max_depth': trial.suggest_int('max_depth', 3, 12)}
    
    model = LGBMClassifier(**params, random_state=RANDOM_STATE, verbose=-1)
    model.fit(xtrain_new, ytrain_new)

    preds = model.predict_proba(xval)[:,1]
    return roc_auc_score(yval, preds)

lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(objective_lgbm_pro, n_trials=30)

[I 2026-04-08 19:54:59,382] A new study created in memory with name: no-name-b820a3b7-7d5f-4988-960b-937a66a66c3c
[I 2026-04-08 19:55:00,206] Trial 0 finished with value: 0.7111299090952681 and parameters: {'num_leaves': 70, 'n_estimators': 240, 'learning_rate': 0.17450714732899933, 'max_depth': 6}. Best is trial 0 with value: 0.7111299090952681.
[I 2026-04-08 19:55:00,764] Trial 1 finished with value: 0.7265362868989825 and parameters: {'num_leaves': 15, 'n_estimators': 404, 'learning_rate': 0.042704193083745824, 'max_depth': 7}. Best is trial 1 with value: 0.7265362868989825.
[I 2026-04-08 19:55:00,982] Trial 2 finished with value: 0.7256137666492022 and parameters: {'num_leaves': 90, 'n_estimators': 187, 'learning_rate': 0.028326766588444782, 'max_depth': 3}. Best is trial 1 with value: 0.7265362868989825.
[I 2026-04-08 19:55:03,984] Trial 3 finished with value: 0.72322517662276 and parameters: {'num_leaves': 81, 'n_estimators': 562, 'learning_rate': 0.06459258760774393, 'max_depth'

In [19]:
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations',100,1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'verbose': False
    }
    
    model = CatBoostClassifier(**params, random_seed=RANDOM_STATE)
    model.fit(xtrain_new, ytrain_new, cat_features=cat_features)

    preds = model.predict_proba(xval)[:,1]
    return roc_auc_score(yval, preds)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(objective_catboost, n_trials=10)

[I 2026-04-08 19:59:15,090] A new study created in memory with name: no-name-c493c68a-d92e-4441-aa19-46f806923f72
[I 2026-04-08 20:00:01,549] Trial 0 finished with value: 0.7536346214232502 and parameters: {'iterations': 606, 'depth': 5, 'learning_rate': 0.18076254298493566}. Best is trial 0 with value: 0.7536346214232502.
[I 2026-04-08 20:01:43,104] Trial 1 finished with value: 0.7410022457199461 and parameters: {'iterations': 830, 'depth': 8, 'learning_rate': 0.18115831365558696}. Best is trial 0 with value: 0.7536346214232502.
[I 2026-04-08 20:03:25,949] Trial 2 finished with value: 0.747522610695486 and parameters: {'iterations': 638, 'depth': 10, 'learning_rate': 0.013199754709576456}. Best is trial 0 with value: 0.7536346214232502.
[I 2026-04-08 20:05:20,767] Trial 3 finished with value: 0.7475591390948823 and parameters: {'iterations': 835, 'depth': 7, 'learning_rate': 0.01369796571866057}. Best is trial 0 with value: 0.7536346214232502.
[I 2026-04-08 20:09:20,268] Trial 4 finis

In [20]:
best_lgbm = LGBMClassifier(**study.best_params, random_state=RANDOM_STATE)
best_lgbm.fit(xtrain, ytrain)

final_lgbm = best_lgbm.predict_proba(xtest)[:, 1]
final_roc_auc = roc_auc_score(ytest, final_preds)
final_roc_auc

0.7325832095846545

In [21]:
best_lgbm = LGBMClassifier(**lgbm_study.best_params, random_state=RANDOM_STATE)
best_lgbm.fit(xtrain, ytrain)
final_lgbm = best_lgbm.predict_proba(xtest)[:, 1]
final_roc_auc = roc_auc_score(ytest, final_lgbm)
final_roc_auc

0.7356378140902512

## Quiz

Поделитесь своими результатами!